# 27. bridge accord 의 검색 성립성

> `spec.md` §8 남은 작업 1번의 선행 확인. 대화에서 "A단계"로 부른 작업이다.

| | |
|---|---|
| 질문 | 노트북 23(v2)이 제안한 accord 로 **실제 검색하면 향수가 몇 개 나오는가** |
| 왜 | `spec.md` §8 남은 작업 1번의 선행 확인. *"accord 92개로 표현을 담을 수 있는가"* |
| API | **호출하지 않는다.** 노트북 23의 저장된 결과를 읽는다 |
| 사람 | **필요 없다.** 의미 판정(B단계)은 이 노트북의 범위가 아니다 |
| 작성 | 2026-09-10 |

## 이 노트북이 답하는 것과 답하지 않는 것

`22_pilot_human_evaluation.csv` 의 사람 판정은 **"번역이 의미상 맞는가"** 를 묻는다.
그런데 의미가 맞아도 검색이 안 될 수 있다.

`비 온 뒤의 숲 → foresty` 는 의미상 정확한 번역이고 사람이 채점하면 만점을 받는다.
그런데 `foresty` 를 가진 향수는 13만 개 중 **1개**다. 사람 점수가 만점이어도 기능은 동작하지 않는다.

사전등록된 ABSTAIN 원인 분류(`APPROPRIATE_ABSTENTION` / `TARGET_SPACE_LIMITATION` /
`MAPPING_FAILURE`)는 셋 모두 *"적절한 target 이 있는가"* 를 묻고,
*"그 target 이 변별력이 있는가"* 를 묻는 칸이 없다.

이 노트북은 그 빠진 축만 계산한다. 사람 판정을 대신하지 않는다.

## 0. 실행 조건과 한계

**API 를 호출하지 않는다.** v2 의 매핑 결과는 `23_..._v2_checkpoint.json` 과
`23_..._v2_comparison.csv` 에 저장돼 있다. `.env` 의 `GMS_KEY` 상태와 무관하게 실행된다.

**기존 노트북을 실행하지 않는다.** 노트북 16 은 재실행하면 QA 판정 60건이 지워지고,
13·14·15·22·23 은 SHA-256 guard 가 걸려 있다. 25·26 번과 같이 새 노트북으로 만든다.

**한계 — 결과를 읽을 때 반드시 함께 볼 것**

- **쿼리 12개다.** 노트북 22 의 사전등록 선정분이며 무작위 표본이 아니다.
  "숲 표현 3건이 전부 `foresty` 로 갔다" 는 12건 중 3건의 이야기다.
- **§7 의 대안 조합은 검증된 매핑이 아니다.** 검색 개수만 확인한다.
  `mossy+earthy` 가 `비 온 뒤의 숲` 의 *옳은* 번역인지는 사람이 판정할 문제다(B단계).
- **계절·시간대 조건을 계산에 넣지 않았다.** `spec.md` 제약 1 대로 서비스 초기에는
  `perfume_review_seasons` 가 0건이다.
- **note 조건을 계산에 넣지 않았다.** accord 만 계산한다.
  `SQ0051` 의 direct note `Peach` 는 결과 표에 `미적용 note 조건` 으로 표시만 한다.
- **점수 식은 검증된 것이 아니다.** `spec.md` §3 의 주의대로 쿼리→향수 점수 식은 미검증이다.
  여기서는 target accord 의 strength 합이라는 가장 단순한 형태만 쓴다.

**하지 않는 것** — 평가 데이터 수정, 기존 노트북 수정·실행, 사전 구축, 사람 판정,
LLM 재호출, 프롬프트 수정.

In [1]:
import hashlib
import itertools
import json
import pathlib

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 44)
pd.set_option("display.width", 220)

# True면 계산과 표시만 하고 파일을 만들지 않는다.
REPORT_ONLY = False

print("REPORT_ONLY:", REPORT_ONLY)

REPORT_ONLY: False


## 1. 경로 · 입력 해싱 · 쓰기 가드

26번과 같은 방식이다. 실행 전후로 입력을 해싱해 대조하고, `write_output()` 으로만 저장한다.

In [2]:
PROJECT_ROOT = pathlib.Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "analysis_outputs"

INPUT_PATHS = {
    "v2_comparison": OUTPUT_DIR / "23_semantic_bridge_accord_only_v2_comparison.csv",
    "pilot_checkpoint": OUTPUT_DIR / "22_semantic_bridge_pilot_checkpoint.json",
    "accord_dictionary": OUTPUT_DIR / "10_accord_dictionary.csv",
    "accord_cooccurrence": OUTPUT_DIR / "10_accord_cooccurrence.csv",
    "perfumes": PROJECT_ROOT / "perfumes.csv",
}
OUTPUT_PATHS = {
    "feasibility": OUTPUT_DIR / "27_bridge_accord_feasibility.csv",
    "alternatives": OUTPUT_DIR / "27_bridge_accord_alternatives.csv",
    "summary": OUTPUT_DIR / "27_bridge_accord_summary.md",
}


def sha256_file(path):
    """파일의 SHA-256 hex digest. str."""
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


missing = [str(p) for p in INPUT_PATHS.values() if not p.is_file()]
if missing:
    raise FileNotFoundError(f"필수 입력이 없습니다: {missing}")
input_hashes_before = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}

FROZEN_FILES = {p.resolve() for p in INPUT_PATHS.values()}
# 이 노트북이 절대 건드리지 않는 것.
PROTECTED_EXTRA = {
    (PROJECT_ROOT / "evaluation_data" / "stage1"
     / "13_stage1_golden_set_v1_200.xlsx").resolve(),
    (PROJECT_ROOT / "evaluation_data" / "stage1"
     / "16_golden_set_quality_audit_reviewed.csv").resolve(),
    (PROJECT_ROOT / "evaluation_data" / "semantic_bridge"
     / "22_pilot_human_evaluation.csv").resolve(),
    (OUTPUT_DIR / "16_golden_set_quality_audit_candidates.csv").resolve(),
}


def write_output(path, writer):
    """OUTPUT_DIR 의 27_ prefix 파일에만 쓴다. REPORT_ONLY면 생략. pathlib.Path 또는 None."""
    path = pathlib.Path(path).resolve()
    if path.parent != OUTPUT_DIR.resolve():
        raise RuntimeError(f"쓰기 허용 디렉터리가 아닙니다: {path}")
    if not path.name.startswith("27_"):
        raise RuntimeError(f"27_ prefix가 아닌 출력은 금지합니다: {path.name}")
    if path in FROZEN_FILES or path in PROTECTED_EXTRA:
        raise RuntimeError(f"보호 파일 덮어쓰기 시도: {path.name}")
    if REPORT_ONLY:
        print(f"[REPORT_ONLY] 저장 생략: {path.name}")
        return None
    writer(path)
    print(f"저장: {path.name}")
    return path


display(pd.Series(input_hashes_before, name="sha256").str.slice(0, 16).to_frame())

,sha256
v2_comparison,e09189d2b02a7dcb
pilot_checkpoint,7af3984421d33690
accord_dictionary,567e91370e575731
accord_cooccurrence,c1d8a80c1f84dc21
perfumes,cec1ea0b49885303


## 2. 사전 등록 — 데이터를 보기 전에 고정한다

계산 규칙과 판정 기준을 데이터 로드보다 앞에 둔다. 결과를 본 뒤 기준을 바꾸지 않기 위해서다.

**§7 의 대안 조합에 대한 정직한 기록** — 이 조합은 팀이 승인한 매핑이 아니다.
`10_accord_dictionary.csv` 에서 코퍼스 지지도가 있는 accord 중 표현과 의미가 통할 후보를
AI 가 초안으로 골랐다. 목적은 **"92개 안에 쓸 수 있는 조합이 존재하는가"** 를 확인하는 것이고,
**"이것이 옳은 번역인가"** 는 이 노트북이 답하지 않는다.

In [3]:
PREREG = {
    "notebook": "27_bridge_accord_search_feasibility",
    "question": "v2 가 제안한 accord 로 검색하면 향수가 몇 개 나오는가",
    "llm_calls": 0,
    "human_judgment": False,

    # spec.md §3 — 결과를 항상 3~5개 돌려준다
    "TOP_K": 5,
    "MIN_RESULTS": 3,

    # 점수 식: target accord 의 strength 합. spec.md §3 의 주의대로 미검증 형태다.
    "score": "sum(perfume_accord_strength for accord in targets)",

    # 판정 기준 (하드 게이트는 이것 하나뿐이다)
    "verdict_rule": "전체 후보 < MIN_RESULTS 이면 검색 불가. 그 외는 검색 가능",

    # 동점 구간은 읽기 편하게 나눈 서술용 구분이며 GO/STOP 기준이 아니다
    "tie_bands": {"후보 부족": "후보 < MIN_RESULTS", "완전 변별": "<= TOP_K",
                  "실용 범위": "<= 50", "순위 임의성 큼": "> 50"},

    # 재현 게이트
    "gates": [
        "10_accord_dictionary.csv 의 perfume_count 92개를 재현",
        "10_accord_cooccurrence.csv 의 cooccurrence_count 로 2개 조합 AND 를 교차검증",
    ],
}

# §7 대안 조합 — AI 초안. 팀 미승인. 의미 판정 없음.
ALTERNATIVE_PROBES = [
    ("SQ0061", "비 온 뒤의 숲의 냄새", "v2 원본", ["foresty", "earthy"]),
    ("SQ0061", "비 온 뒤의 숲의 냄새", "대안", ["mossy", "earthy"]),
    ("SQ0061", "비 온 뒤의 숲의 냄새", "대안", ["mossy", "earthy", "green"]),
    ("SQ0032", "숲 속에 온 듯한 향", "v2 원본", ["foresty"]),
    ("SQ0032", "숲 속에 온 듯한 향", "대안", ["mossy", "green"]),
    ("SQ0032", "숲 속에 온 듯한 향", "대안", ["woody", "green"]),
    ("SQ0073", "편백나무·산내음 (+direct)", "v2 원본", ["woody", "floral", "fresh", "foresty"]),
    ("SQ0073", "편백나무·산내음 (+direct)", "대안", ["woody", "conifer", "green"]),
    ("SQ0073", "편백나무·산내음 (+direct)", "대안", ["woody", "conifer", "fresh"]),
    ("SQ0132", "포근한 느낌 (+direct)", "v2 ABSTAIN → direct만", ["musky"]),
    ("SQ0132", "포근한 느낌 (+direct)", "spec §3 예시", ["musky", "powdery", "vanilla"]),
    ("SQ0132", "포근한 느낌 (+direct)", "spec §3 예시 bridge만", ["powdery", "vanilla"]),
    ("SQ0002", "시원하고 깔끔한 향 (+direct)", "v2 원본", ["citrus", "fresh"]),
    ("SQ0002", "시원하고 깔끔한 향 (+direct)", "번역 실패 가정", ["citrus"]),
]

TOP_K = PREREG["TOP_K"]
MIN_RESULTS = PREREG["MIN_RESULTS"]
display(Markdown("```json\n" + json.dumps(PREREG, ensure_ascii=False, indent=2) + "\n```"))

```json
{
  "notebook": "27_bridge_accord_search_feasibility",
  "question": "v2 가 제안한 accord 로 검색하면 향수가 몇 개 나오는가",
  "llm_calls": 0,
  "human_judgment": false,
  "TOP_K": 5,
  "MIN_RESULTS": 3,
  "score": "sum(perfume_accord_strength for accord in targets)",
  "verdict_rule": "전체 후보 < MIN_RESULTS 이면 검색 불가. 그 외는 검색 가능",
  "tie_bands": {
    "후보 부족": "후보 < MIN_RESULTS",
    "완전 변별": "<= TOP_K",
    "실용 범위": "<= 50",
    "순위 임의성 큼": "> 50"
  },
  "gates": [
    "10_accord_dictionary.csv 의 perfume_count 92개를 재현",
    "10_accord_cooccurrence.csv 의 cooccurrence_count 로 2개 조합 AND 를 교차검증"
  ]
}
```

## 3. accord 파싱

`perfumes.csv` 의 `accords` 컬럼은 `name:strength|name:strength|...` 형식이다
(`SCHEMA.md`: `accords[].strength` 는 0~100).

accord 이름에 공백이 있으므로(`fresh spicy`, `warm spicy`) `rpartition(":")` 으로 자른다.
`split(":")` 을 쓰면 이름이 잘린다.

In [4]:
perfume_df = pd.read_csv(INPUT_PATHS["perfumes"], usecols=["id", "accords"],
                         dtype={"id": str})
n_rows = len(perfume_df)
n_blank = int(perfume_df.accords.fillna("").eq("").sum())

PERFUME_ACCORDS = []          # [{accord: strength}] — accord 를 가진 향수만
malformed = 0
for raw in perfume_df.accords.fillna(""):
    if not raw:
        continue
    parsed = {}
    for part in raw.split("|"):
        if not part:
            continue
        name, sep, strength = part.rpartition(":")
        if not sep or not strength.isdigit():
            malformed += 1
            continue
        parsed[name] = int(strength)
    if parsed:
        PERFUME_ACCORDS.append(parsed)

STRENGTHS = {}
for row in PERFUME_ACCORDS:
    for accord, strength in row.items():
        STRENGTHS.setdefault(accord, []).append(strength)
STRENGTHS = {a: np.array(v) for a, v in STRENGTHS.items()}

print(f"perfumes.csv 행                {n_rows:,}")
print(f"  accords 빈 값                {n_blank:,}  (제외)")
print(f"  accord 보유 향수             {len(PERFUME_ACCORDS):,}")
print(f"  파싱 실패한 accord 조각      {malformed}")
print(f"  서로 다른 accord             {len(STRENGTHS)}")

perfumes.csv 행                131,930
  accords 빈 값                2,769  (제외)
  accord 보유 향수             129,161
  파싱 실패한 accord 조각      0
  서로 다른 accord             92


## 4. 재현 게이트

파싱이 기존 산출물과 어긋나면 이후 숫자를 믿을 수 없다. 두 가지를 대조하고 실패하면 중단한다.

1. accord 별 보유 향수 수 → `10_accord_dictionary.csv` 의 `perfume_count`
2. 2개 조합의 AND 개수 → `10_accord_cooccurrence.csv` 의 `cooccurrence_count`

In [5]:
accord_dict = pd.read_csv(INPUT_PATHS["accord_dictionary"])
cooc = pd.read_csv(INPUT_PATHS["accord_cooccurrence"])


def and_count(targets):
    """target accord 를 모두 가진 향수 수. int."""
    return sum(1 for row in PERFUME_ACCORDS if all(t in row for t in targets))


# --- 게이트 1
ref_count = dict(zip(accord_dict.accord, accord_dict.perfume_count))
gate1 = [(a, ref_count[a], len(STRENGTHS.get(a, [])))
         for a in ref_count if len(STRENGTHS.get(a, [])) != ref_count[a]]
if gate1:
    for accord, expected, got in gate1[:10]:
        print(f"  {accord:16s} 기존 {expected:>7,}  파싱 {got:>7,}")
    raise RuntimeError(f"재현 게이트 1 실패 — {len(gate1)}개 accord 불일치")
print(f"게이트 1 통과 — accord {len(ref_count)}개 perfume_count 전부 일치 (오차 0)")

# --- 게이트 2
COOC = {frozenset((r.accord, r.related_accord)): r.cooccurrence_count
        for r in cooc.itertuples(index=False)}

probe_pairs = {frozenset(pair)
               for _, _, _, targets in ALTERNATIVE_PROBES
               for pair in itertools.combinations(sorted(set(targets)), 2)}
checkable = sorted(probe_pairs & set(COOC), key=lambda s: sorted(s))
gate2 = [(tuple(sorted(p)), COOC[p], and_count(list(p))) for p in checkable
         if and_count(list(p)) != COOC[p]]
if gate2:
    for pair, expected, got in gate2[:10]:
        print(f"  {pair} 기존 {expected:>7,}  파싱 {got:>7,}")
    raise RuntimeError(f"재현 게이트 2 실패 — {len(gate2)}개 조합 불일치")
print(f"게이트 2 통과 — 2개 조합 {len(checkable)}건 cooccurrence_count 전부 일치 (오차 0)")

게이트 1 통과 — accord 92개 perfume_count 전부 일치 (오차 0)


게이트 2 통과 — 2개 조합 14건 cooccurrence_count 전부 일치 (오차 0)


## 5. 검색 개수와 순위 변별력

두 가지를 함께 낸다.

- **후보** — target accord 를 **모두** 가진 향수 수. `spec.md` §3 의 `required: true` AND 조건
- **5위 동점** — strength 합으로 정렬했을 때 상위 5위에 들려면 몇 개와 경쟁하는가

두 번째가 중요하다. 후보가 33,478개여도 그중 5개를 고를 수 있으면 기능은 동작한다.
반대로 후보가 많으면서 같은 점수가 수천 개면 상위 5개는 제비뽑기가 된다.

In [6]:
def search_probe(targets, k=TOP_K):
    """target accord 조합의 후보 수와 상위 k위 진입 동점 규모. dict."""
    targets = list(targets)
    if not targets:
        return {"후보": None, "5위 점수": None, "5위 동점": None}
    scores = [sum(row[t] for t in targets)
              for row in PERFUME_ACCORDS if all(t in row for t in targets)]
    n = len(scores)
    if n == 0:
        return {"후보": 0, "5위 점수": None, "5위 동점": 0}
    if n <= k:
        return {"후보": n, "5위 점수": min(scores), "5위 동점": n}
    cut = sorted(scores, reverse=True)[k - 1]
    return {"후보": n, "5위 점수": cut, "5위 동점": sum(1 for s in scores if s >= cut)}


def tie_band(tie, n_candidates=None):
    """5위 동점 규모를 서술용 구간으로 바꾼다. str.

    후보 자체가 MIN_RESULTS 미만이면 동점 규모를 논할 수 없으므로 먼저 구분한다.
    `foresty` 처럼 후보 1개인 경우가 '완전 변별'로 읽히는 것을 막는다.
    """
    if tie is None:
        return ""
    if n_candidates is not None and n_candidates < MIN_RESULTS:
        return "후보 부족"
    if tie <= TOP_K:
        return "완전 변별"
    if tie <= 50:
        return "실용 범위"
    return "순위 임의성 큼"


def verdict(status, combined_n):
    """쿼리 단위 판정. str."""
    if status != "MAPPED":
        return "번역 없음"
    if combined_n is None:
        return "검색 불가 (조건 없음)"
    if combined_n < MIN_RESULTS:
        return "검색 불가"
    return "검색 가능"


v2_df = pd.read_csv(INPUT_PATHS["v2_comparison"], keep_default_na=False, dtype=str)
pilot_ckpt = json.loads(INPUT_PATHS["pilot_checkpoint"].read_text(encoding="utf-8"))
direct_features = {q: v["baseline_features"]
                   for q, v in pilot_ckpt["retrieval_key"].items()}

rows = []
for record in v2_df.to_dict("records"):
    qid = record["query_id"]
    bridge = json.loads(record["v2_accords_json"])
    direct = list(direct_features[qid]["accords"])
    dnotes = list(direct_features[qid]["canonical_notes"])
    combined = list(dict.fromkeys(direct + bridge))

    bridge_probe = search_probe(bridge)
    full_probe = search_probe(combined)
    rows.append({
        "query_id": qid,
        "구분": record["resolution_type"],
        "표현": record["bridge_phrase"],
        "v2 상태": record["v2_status"],
        "bridge accord": "+".join(bridge),
        "direct accord": "+".join(direct),
        "미적용 note 조건": "+".join(dnotes),
        "bridge 후보": bridge_probe["후보"],
        "bridge 5위 동점": bridge_probe["5위 동점"],
        "전체 후보": full_probe["후보"],
        "전체 5위 동점": full_probe["5위 동점"],
        "동점 구간": tie_band(full_probe["5위 동점"], full_probe["후보"]),
        "판정": verdict(record["v2_status"], full_probe["후보"]),
    })

feasibility_df = pd.DataFrame(rows)
for col in ["bridge 후보", "bridge 5위 동점", "전체 후보", "전체 5위 동점"]:
    feasibility_df[col] = feasibility_df[col].astype("Int64")
display(feasibility_df[[
    "query_id", "구분", "bridge accord", "direct accord",
    "bridge 후보", "전체 후보", "전체 5위 동점", "동점 구간", "판정",
]])

,query_id,구분,bridge accord,direct accord,bridge 후보,전체 후보,전체 5위 동점,동점 구간,판정
0,SQ0061,PURE,foresty+earthy,,0,0,0,후보 부족,검색 불가
1,SQ0032,PURE,foresty,,1,1,1,후보 부족,검색 불가
2,SQ0136,PURE,fresh+soapy,,1049,1049,6,실용 범위,검색 가능
3,SQ0105,PURE,,,<NA>,<NA>,<NA>,,번역 없음
4,SQ0012,PURE,soapy+warm spicy,,114,114,5,완전 변별,검색 가능
5,SQ0058,PURE,aquatic+fresh,,4963,4963,14,실용 범위,검색 가능
6,SQ0002,MIXED,fresh,citrus,33478,18826,25,실용 범위,검색 가능
7,SQ0047,MIXED,fresh+citrus,soapy+musky+woody,18826,13,5,완전 변별,검색 가능
8,SQ0051,MIXED,fresh+soapy,,1049,1049,6,실용 범위,검색 가능
9,SQ0071,MIXED,fresh+citrus,musky+woody,18826,2540,5,완전 변별,검색 가능


### 판정 집계

`spec.md` §3 은 결과를 항상 3~5개 돌려준다고 정의한다. 후보가 3개 미만이면
조건 완화로도 메울 수 없는 실패다.

In [7]:
display(feasibility_df["판정"].value_counts().rename("쿼리 수").to_frame())

blocked = feasibility_df[feasibility_df["판정"].str.startswith("검색 불가")]
if len(blocked):
    print("검색 불가 쿼리와 지지도가 낮은 accord")
    for record in blocked.to_dict("records"):
        thin = [f"{a}({len(STRENGTHS[a]):,})"
                for a in record["bridge accord"].split("+")
                if a and len(STRENGTHS[a]) < 1000]
        print(f"  {record['query_id']}  {record['표현'][:28]:30s} "
              f"{record['bridge accord']:26s} 후보 {record['전체 후보']}개   {thin}")

,쿼리 수
판정,
검색 가능,7
검색 불가,3
번역 없음,2


검색 불가 쿼리와 지지도가 낮은 accord
  SQ0061  비 온 뒤의 숲의 냄새                   foresty+earthy             후보 0개   ['foresty(1)']
  SQ0032  숲 속에 온 듯한 향                    foresty                    후보 1개   ['foresty(1)']
  SQ0073  깨끗하고 시원한 향을 좋아하고, 편백나무, 산내음처   fresh+foresty              후보 0개   ['foresty(1)']


## 6. 쓰인 accord 의 코퍼스 지지도와 강도 분포

`foresty` 가 향수 1개라는 사실은 **LLM 이 구조적으로 알 수 없다.** 세상 지식이 아니라
우리 데이터 안에만 있는 사실이기 때문이다(측정 기록 1번).

`corpus_support` 를 Domain Lexicon 의 필수 컬럼으로 둔 근거가 이 표다.

In [8]:
used = {}
for record in feasibility_df.to_dict("records"):
    for accord in record["bridge accord"].split("+"):
        if accord:
            used[accord] = used.get(accord, 0) + 1

strength_rows = []
for accord, n_query_used in sorted(used.items(), key=lambda kv: -len(STRENGTHS[kv[0]])):
    s = STRENGTHS[accord]
    solo = search_probe([accord])
    strength_rows.append({
        "accord": accord,
        "쓰인 쿼리": n_query_used,
        "보유 향수": len(s),
        "전체 비율": f"{len(s) / len(PERFUME_ACCORDS):.1%}",
        "중앙 강도": int(np.median(s)),
        "p90 강도": int(np.percentile(s, 90)),
        "최대 강도": int(s.max()),
        "최대값 동점": int((s == s.max()).sum()),
        "단독 5위 동점": solo["5위 동점"],
        "단독 동점 구간": tie_band(solo["5위 동점"], solo["후보"]),
    })
strength_df = pd.DataFrame(strength_rows)
display(strength_df)

,accord,쓰인 쿼리,보유 향수,전체 비율,중앙 강도,p90 강도,최대 강도,최대값 동점,단독 5위 동점,단독 동점 구간
0,citrus,2,59969,46.4%,67,100,100,17736,17736,순위 임의성 큼
1,warm spicy,1,45319,35.1%,52,100,100,6281,6281,순위 임의성 큼
2,fresh,7,33478,25.9%,40,72,100,830,830,순위 임의성 큼
3,earthy,1,19049,14.7%,40,71,100,533,533,순위 임의성 큼
4,aquatic,1,8872,6.9%,41,90,100,609,609,순위 임의성 큼
5,soapy,3,1720,1.3%,20,30,100,18,18,실용 범위
6,foresty,3,1,0.0%,44,44,44,1,1,후보 부족


## 7. 대안 조합이 92개 안에 있는가

검색 불가로 판정된 쿼리에 대해, **같은 92개 목록 안의 다른 accord** 로 바꾸면
검색이 되는지 본다.

**다시 강조 — 여기서 보는 것은 "검색이 되는가" 뿐이다.**
`mossy+earthy` 가 `비 온 뒤의 숲` 의 옳은 번역인지는 이 노트북이 답하지 않는다.
사전 등록(§2)에 적은 대로 이 조합은 AI 초안이며 팀이 승인한 매핑이 아니다.

In [9]:
alt_rows = []
for qid, phrase, label, targets in ALTERNATIVE_PROBES:
    probe = search_probe(targets)
    alt_rows.append({
        "query_id": qid,
        "표현": phrase,
        "구분": label,
        "accord 조합": "+".join(targets),
        "후보": probe["후보"],
        "5위 점수": probe["5위 점수"],
        "5위 동점": probe["5위 동점"],
        "동점 구간": tie_band(probe["5위 동점"], probe["후보"]),
        "검색 가능": "예" if (probe["후보"] or 0) >= MIN_RESULTS else "아니오",
    })
alternatives_df = pd.DataFrame(alt_rows)
for col in ["후보", "5위 점수", "5위 동점"]:
    alternatives_df[col] = alternatives_df[col].astype("Int64")
display(alternatives_df)

,query_id,표현,구분,accord 조합,후보,5위 점수,5위 동점,동점 구간,검색 가능
0,SQ0061,비 온 뒤의 숲의 냄새,v2 원본,foresty+earthy,0,<NA>,0,후보 부족,아니오
1,SQ0061,비 온 뒤의 숲의 냄새,대안,mossy+earthy,4715,200,8,실용 범위,예
2,SQ0061,비 온 뒤의 숲의 냄새,대안,mossy+earthy+green,1254,286,5,완전 변별,예
3,SQ0032,숲 속에 온 듯한 향,v2 원본,foresty,1,44,1,후보 부족,아니오
4,SQ0032,숲 속에 온 듯한 향,대안,mossy+green,1702,200,20,실용 범위,예
5,SQ0032,숲 속에 온 듯한 향,대안,woody+green,17714,200,121,순위 임의성 큼,예
6,SQ0073,편백나무·산내음 (+direct),v2 원본,woody+floral+fresh+foresty,0,<NA>,0,후보 부족,아니오
7,SQ0073,편백나무·산내음 (+direct),대안,woody+conifer+green,720,235,5,완전 변별,예
8,SQ0073,편백나무·산내음 (+direct),대안,woody+conifer+fresh,459,240,5,완전 변별,예
9,SQ0132,포근한 느낌 (+direct),v2 ABSTAIN → direct만,musky,41154,100,4010,순위 임의성 큼,예


## 8. 결론

아래 숫자는 모두 위 셀의 출력에서 가져온다. 손으로 적지 않는다.

In [10]:
n_query = len(feasibility_df)
n_mapped = int((feasibility_df["v2 상태"] == "MAPPED").sum())
n_ok = int((feasibility_df["판정"] == "검색 가능").sum())
n_blocked = int(feasibility_df["판정"].str.startswith("검색 불가").sum())
n_abstain = int((feasibility_df["판정"] == "번역 없음").sum())

dead_accords = sorted({accord
                       for record in feasibility_df.to_dict("records")
                       for accord in record["bridge accord"].split("+")
                       if accord and len(STRENGTHS[accord]) < MIN_RESULTS})
dead_used_in = int(sum(1 for record in feasibility_df.to_dict("records")
                       if any(a in dead_accords
                              for a in record["bridge accord"].split("+") if a)))
solo_bad = strength_df[strength_df["단독 동점 구간"] == "순위 임의성 큼"]
fixed = alternatives_df[(alternatives_df["구분"] == "대안")
                        & (alternatives_df["검색 가능"] == "예")]

display(Markdown(f"""
| | |
|---|---:|
| 쿼리 | {n_query} |
| v2 가 번역한 쿼리 | {n_mapped} |
| 검색 가능 | **{n_ok}** |
| 검색 불가 | **{n_blocked}** |
| 번역 없음 (ABSTAIN) | {n_abstain} |
| 검색 불가의 원인 accord | `{"`, `".join(dead_accords)}` |
| 그 accord 가 쓰인 쿼리 | {dead_used_in} |
| 단독 사용 시 순위 임의성이 큰 accord | {len(solo_bad)}종 |
| 92개 안의 대안으로 검색이 된 조합 | {len(fixed)}건 |
"""))

display(solo_bad[["accord", "보유 향수", "최대값 동점", "단독 5위 동점"]])


| | |
|---|---:|
| 쿼리 | 12 |
| v2 가 번역한 쿼리 | 10 |
| 검색 가능 | **7** |
| 검색 불가 | **3** |
| 번역 없음 (ABSTAIN) | 2 |
| 검색 불가의 원인 accord | `foresty` |
| 그 accord 가 쓰인 쿼리 | 3 |
| 단독 사용 시 순위 임의성이 큰 accord | 5종 |
| 92개 안의 대안으로 검색이 된 조합 | 6건 |


,accord,보유 향수,최대값 동점,단독 5위 동점
0,citrus,59969,17736,17736
1,warm spicy,45319,6281,6281
2,fresh,33478,830,830
3,earthy,19049,533,533
4,aquatic,8872,609,609


## 9. 저장

In [11]:
band_counts = feasibility_df["동점 구간"].value_counts().to_dict()
solo_table = "\n".join(
    f"| `{r['accord']}` | {r['보유 향수']:,} | {r['중앙 강도']} | "
    f"{r['최대값 동점']:,} | {r['단독 5위 동점']:,} |"
    for r in strength_df.to_dict("records"))
alt_table = "\n".join(
    f"| {r['query_id']} | {r['구분']} | `{r['accord 조합']}` | {r['후보']:,} | "
    f"{'' if r['5위 동점'] is None else format(r['5위 동점'], ',')} | {r['검색 가능']} |"
    for r in alternatives_df.to_dict("records"))
thinnest = min(len(STRENGTHS[a]) for a in dead_accords) if dead_accords else None

summary_text = f"""# bridge accord 의 검색 성립성

## 질문

노트북 23(v2)이 제안한 accord 로 실제 검색하면 향수가 몇 개 나오는가.
`spec.md` §8 남은 작업 1번(*"accord 92개로 표현을 담을 수 있는가"*)의 선행 확인이다.

## 왜 사람 판정만으로는 답이 안 나오는가

`22_pilot_human_evaluation.csv` 의 사람 판정은 **번역이 의미상 맞는가**를 묻는다.
`비 온 뒤의 숲 → foresty` 는 의미상 정확하고 채점하면 만점을 받는다.
그런데 `foresty` 보유 향수는 **{thinnest}개**다. 사람 점수가 만점이어도 기능은 동작하지 않는다.

사전등록된 ABSTAIN 원인 분류 세 개는 모두 *"적절한 target 이 있는가"* 를 묻고,
*"그 target 이 변별력이 있는가"* 를 묻는 칸이 없다. 이 노트북은 그 축만 계산한다.

## 설계

- **API 호출 0회.** 노트북 23 의 저장된 매핑을 읽는다
- **사람 판정 0건**
- 점수 = target accord 의 strength 합 (`spec.md` §3 의 주의대로 미검증 형태)
- 재현 게이트 2개를 먼저 통과시킨다
  - accord {len(accord_dict)}개 `perfume_count` 재현 — 오차 0
  - 2개 조합 {len(checkable)}건 `cooccurrence_count` 교차검증 — 오차 0
- 입력 {len(PERFUME_ACCORDS):,}개 향수 (전체 {n_rows:,}개 중 accord 없는 {n_blank:,}개 제외)

## 결과

| | |
|---|---:|
| 쿼리 | {n_query} |
| v2 가 번역한 쿼리 | {n_mapped} |
| **검색 가능** | **{n_ok}** |
| **검색 불가** | **{n_blocked}** |
| 번역 없음 (ABSTAIN) | {n_abstain} |

검색 불가의 원인은 전부 `{"`, `".join(dead_accords)}` 하나다. 보유 향수가 {thinnest}개이므로
다른 accord 와 AND 로 묶으면 0이 된다. 그리고 이 accord 는 v2 가 번역한 {n_mapped}건 중
**{dead_used_in}건**에 쓰였다 — 숲·자연 표현 전부다.

## 조합은 되고 단독은 안 된다

| accord | 보유 향수 | 중앙 강도 | 최대값 동점 | 단독 5위 동점 |
|---|---:|---:|---:|---:|
{solo_table}

넓은 accord 두 개를 AND 로 묶고 strength 합으로 정렬하면 순위가 생긴다.
단독으로 쓰면 같은 점수가 수천 개라 상위 5개가 제비뽑기가 된다.

전체 조건 기준 동점 구간 분포: {band_counts}

**사전 설계 규칙 하나가 여기서 나온다 — 한 표현은 accord 2개 이상으로 매핑한다.**
`spec.md` §4.3 의 `required` 컬럼(`core`/`optional`)이 이미 이것을 지원한다.

## 92개 안에 대안이 있다

| query | 구분 | 조합 | 후보 | 5위 동점 | 검색 가능 |
|---|---|---|---:|---:|---|
{alt_table}

`SQ0073` 의 사용자 표현은 **편백나무**이고 92개 안에 **`conifer`(침엽수)** 가 있다.
글자 그대로 대응하는데 v2 는 `foresty` 를 골랐다.
사전등록 §6.4 의 분류로는 `TARGET_SPACE_LIMITATION` 이 아니라 `MAPPING_FAILURE` 다.

**왜 못 찾았는지도 설명된다.** `foresty` 가 향수 {thinnest}개라는 것은 우리 데이터 안에만 있는
사실이므로 LLM 이 알 방법이 없다. 측정 기록 1번의 문장이 그대로 적용된다 —
*"이 정보는 LLM 이 구조적으로 알 수 없다."*

## 그래서 원래 질문의 답

**accord 92개로 담을 수 있다. 조건 두 개를 지키면.**

1. **조합으로 쓴다.** 단독 매핑 금지 → `required` 컬럼이 이미 지원한다
2. **코퍼스 지지도로 죽은 accord 를 배제한다** → `corpus_support` 컬럼이 이미 필수다

두 조건 모두 `spec.md` §4.3 의 사전이 이미 하려던 일이다.
**따라서 사전이 accord 만 겨냥하는 설계를 재검토할 필요는 없다.**
canonical note 를 다시 넣지 않아도 된다(노트북 23 의 축소 판단은 유지된다).

## 한계

- **쿼리 {n_query}개다.** 노트북 22 의 사전등록 선정분이며 무작위 표본이 아니다.
  "숲 표현 {dead_used_in}건이 전부 `foresty` 로 갔다"는 {n_query}건 중 {dead_used_in}건의 이야기다
- **대안 조합은 검증된 매핑이 아니다.** 검색 개수만 확인했다.
  의미 판정은 B단계의 질문이며 이 노트북은 답하지 않는다
- 계절·시간대 조건을 넣지 않았다 (`spec.md` 제약 1 — 서비스 초기 0건)
- note 조건을 넣지 않았다. `SQ0051` 의 direct note `Peach` 는 표시만 했다
- 점수 식이 검증되지 않았다. strength 합이라는 가장 단순한 형태만 썼다
- 단일 실행이다

## 재현 방법

```bash
cd EDA
export PYTHONIOENCODING=utf-8
# 27_bridge_accord_search_feasibility.ipynb 를 REPORT_ONLY=False 로 실행
# API 호출 없음. 재현 게이트 2개가 먼저 통과해야 진행된다
```

## 관련 자료

- `23_semantic_bridge_accord_only_v2_comparison.csv` — v2 매핑 (입력)
- `22_semantic_bridge_pilot_checkpoint.json` — direct annotation (입력)
- `docs/nlr_engineering_notes.md` 1번 — `foresty` 코퍼스 지지도 문제의 최초 발견
- `docs/nlr_engineering_notes.md` 2번 — 단독 accord 의 변별력 문제 (`머스크 → musky`)
- `docs/spec.md` §4.3 — 사전 스키마의 `corpus_support` · `required`
- `docs/plans/SEMANTIC_BRIDGE_PILOT_PLAN.md` §6.4 — ABSTAIN 원인 분류
"""

write_output(OUTPUT_PATHS["feasibility"],
             lambda p: feasibility_df.to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["alternatives"],
             lambda p: alternatives_df.to_csv(p, index=False, encoding="utf-8-sig"))
write_output(OUTPUT_PATHS["summary"],
             lambda p: p.write_text(summary_text, encoding="utf-8"))

if not REPORT_ONLY:
    reread = pd.read_csv(OUTPUT_PATHS["feasibility"])
    if len(reread) != n_query:
        raise ValueError(f"저장 결과 행 수 불일치: {len(reread)} != {n_query}")
    print(f"검증 통과 — feasibility {len(reread)}행 / "
          f"alternatives {len(alternatives_df)}행")

저장: 27_bridge_accord_feasibility.csv
저장: 27_bridge_accord_alternatives.csv
저장: 27_bridge_accord_summary.md
검증 통과 — feasibility 12행 / alternatives 14행


## 10. 가드 검증

입력이 실행 중에 바뀌지 않았는지 확인한다. 보호 대상 평가 데이터도 함께 확인한다.

In [12]:
input_hashes_after = {k: sha256_file(v) for k, v in INPUT_PATHS.items()}
changed = [k for k in input_hashes_before
           if input_hashes_before[k] != input_hashes_after[k]]
if changed:
    raise RuntimeError(f"입력이 변경됐습니다: {changed}")
print("입력 해시 동일")

for path in sorted(PROTECTED_EXTRA):
    print(f"보호 대상 미변경 확인: {path.name}  {'존재' if path.is_file() else '없음'}")

print("생성한 출력:")
for label, path in OUTPUT_PATHS.items():
    mark = "" if path.is_file() else "  (REPORT_ONLY로 미생성)"
    print(f"  {label}: {path.name}{mark}")

입력 해시 동일
보호 대상 미변경 확인: 16_golden_set_quality_audit_candidates.csv  존재
보호 대상 미변경 확인: 22_pilot_human_evaluation.csv  존재
보호 대상 미변경 확인: 13_stage1_golden_set_v1_200.xlsx  존재
보호 대상 미변경 확인: 16_golden_set_quality_audit_reviewed.csv  존재
생성한 출력:
  feasibility: 27_bridge_accord_feasibility.csv
  alternatives: 27_bridge_accord_alternatives.csv
  summary: 27_bridge_accord_summary.md
